# 📊 Principal Component Analysis (PCA)
## Lecture 22 — Dimensionality Reduction in Machine Learning
**By Zamir Hussain | January 20, 2026**

---

### 🚀 How to Run This Notebook
- **VS Code**: Install the `Jupyter` extension → Open this `.ipynb` file → Run cells with `Shift+Enter`
- **Google Colab**: Go to [colab.research.google.com](https://colab.research.google.com) → File → Upload Notebook → Upload this file

---

## 📦 Step 0: Install & Import Libraries
Run this cell first every time!

In [ ]:
# Run this cell first — installs/imports everything needed
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA as SklearnPCA

# Pretty print
np.set_printoptions(precision=4, suppress=True)
print('✅ Libraries loaded successfully!')

---
## 🔢 Part 1: PCA from Scratch — 4 Examples, 2 Features
*(Matches Section 6–8 of the lecture notes)*

**Dataset:**
| Example | X1 | X2 |
|---------|-------|-------|
| 1 | 4 | 11 |
| 2 | 8 | 4 |
| 3 | 13 | 5 |
| 4 | 7 | 14 |

**Goal:** Reduce from 2D → 1D using PCA

In [ ]:
# ─────────────────────────────────────────
# STEP 1: Define the Dataset
# ─────────────────────────────────────────
X = np.array([
    [4,  11],
    [8,   4],
    [13,  5],
    [7,  14]
], dtype=float)

print('Dataset X (N=4, n=2 features):')
print(X)

In [ ]:
# ─────────────────────────────────────────
# STEP 2: Compute Feature Means
# Formula: x̄ᵢ = (1/N) * Σ xₖᵢ
# ─────────────────────────────────────────
mean_vec = np.mean(X, axis=0)   # axis=0 → mean across rows (per column)

print(f'Mean of X1 (X̄₁): {mean_vec[0]}')
print(f'Mean of X2 (X̄₂): {mean_vec[1]}')
print(f'Mean vector X̄  : {mean_vec}')

In [ ]:
# ─────────────────────────────────────────
# STEP 3: Compute the Covariance Matrix
# Formula: Cov(i,j) = 1/(N-1) * Σ(xᵢ - x̄ᵢ)(xⱼ - x̄ⱼ)
# ─────────────────────────────────────────

# Manual calculation
N = X.shape[0]
X_centered = X - mean_vec

# Covariance matrix = (1/(N-1)) * Xᶜ.T @ Xᶜ
S = (X_centered.T @ X_centered) / (N - 1)

print('Mean-Centered Data (X - X̄):')
print(X_centered)
print()
print('Covariance Matrix S (2×2):')
print(S)
print()
print(f'  Var(X1)      = Cov(X1,X1) = {S[0,0]}')
print(f'  Var(X2)      = Cov(X2,X2) = {S[1,1]}')
print(f'  Cov(X1,X2)   = Cov(X2,X1) = {S[0,1]}')

In [ ]:
# ─────────────────────────────────────────
# STEP 4: Compute Eigenvalues & Eigenvectors
# Solve: |S - λI| = 0
# ─────────────────────────────────────────
eigenvalues, eigenvectors = np.linalg.eig(S)

print('Eigenvalues (unsorted):', eigenvalues)
print('Eigenvectors (columns = each eigenvector):')
print(eigenvectors)

In [ ]:
# ─────────────────────────────────────────
# STEP 5: Sort Eigenvalues (Highest → Lowest)
# The largest eigenvalue → most variance captured
# ─────────────────────────────────────────
sorted_idx    = np.argsort(eigenvalues)[::-1]   # descending order
eigenvalues   = eigenvalues[sorted_idx]
eigenvectors  = eigenvectors[:, sorted_idx]      # reorder columns

print('Sorted Eigenvalues (λ₁ ≥ λ₂):')
for i, lam in enumerate(eigenvalues):
    pct = lam / eigenvalues.sum() * 100
    print(f'  λ{i+1} = {lam:.4f}  →  explains {pct:.1f}% of variance')

print()
print('Corresponding Eigenvectors (normalized by numpy):')
print(eigenvectors)

In [ ]:
# ─────────────────────────────────────────
# STEP 6: Select Top p=1 Principal Component
# W = [e₁]  (just first eigenvector column)
# ─────────────────────────────────────────
p = 1   # reduce to 1D
W = eigenvectors[:, :p]   # projection matrix (n × p)

print(f'Projection Matrix W (n×p = 2×{p}):')
print(W)
print()
# Manual eigenvector from lecture for comparison
e1_lecture = np.array([0.5574, -0.8303])
print('Lecture note eigenvector e₁:', e1_lecture)

In [ ]:
# ─────────────────────────────────────────
# STEP 7: Project Data → New 1D Feature Space
# Formula: Y = (X - X̄) × W
# ─────────────────────────────────────────
Y = X_centered @ W    # (N×n) @ (n×p) = (N×p)

print('Transformed 1D Dataset Y (PC1 scores):')
print('-' * 35)
print(f'{"Example":<10} {"X1":<6} {"X2":<6} {"PC1 Score"}')
print('-' * 35)
for i in range(N):
    print(f'{i+1:<10} {X[i,0]:<6.0f} {X[i,1]:<6.0f} {Y[i,0]:.4f}')
print()
print('✅ 2D data successfully reduced to 1D!')

In [ ]:
# ─────────────────────────────────────────
# 📊 VISUALIZATION: Original 2D + PC Direction
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

# Left: Original 2D data
ax1 = axes[0]
for i in range(N):
    ax1.scatter(X[i,0], X[i,1], color=colors[i], s=120, zorder=5)
    ax1.annotate(f'P{i+1}({X[i,0]:.0f},{X[i,1]:.0f})',
                 (X[i,0], X[i,1]), textcoords='offset points',
                 xytext=(8,5), fontsize=9)

# Draw PC1 direction through the mean
scale = 5
pc1 = W[:, 0]
ax1.annotate('', xy=mean_vec + scale*pc1, xytext=mean_vec - scale*pc1,
             arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax1.scatter(*mean_vec, color='black', s=80, marker='x', zorder=6, label='Mean')
ax1.set_title('Original 2D Data + PC1 Direction', fontsize=12, fontweight='bold')
ax1.set_xlabel('X1'); ax1.set_ylabel('X2')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Right: Projected 1D
ax2 = axes[1]
for i in range(N):
    ax2.scatter(Y[i,0], 0, color=colors[i], s=120, zorder=5)
    ax2.annotate(f'P{i+1}\n{Y[i,0]:.3f}',
                 (Y[i,0], 0), textcoords='offset points',
                 xytext=(0, 12), fontsize=9, ha='center')
ax2.axhline(0, color='gray', lw=1)
ax2.set_title('Projected 1D Data (PC1 Scores)', fontsize=12, fontweight='bold')
ax2.set_xlabel('PC1 Score'); ax2.set_yticks([])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/pca_part1_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved!')

---
## 🔢 Part 2: PCA — 3 Examples (Perfect Line Case)
*(Matches Sections 14–16 of the lecture notes)*

**Dataset:** 3 samples, 2 features — data lies perfectly on a line!
| Example | X | Y |
|---------|---|----|
| 1 | 2 | 11 |
| 2 | 3 | 14 |
| 3 | 7 | 26 |

In [ ]:
# ─────────────────────────────────────────
# PART 2: Dataset with N=3, n=2
# ─────────────────────────────────────────
X2 = np.array([[2, 11],
               [3, 14],
               [7, 26]], dtype=float)

# Step 1: Means and centering
mean2  = np.mean(X2, axis=0)
Z2     = X2 - mean2   # centered data

print('Means:', mean2)
print('Centered Data Z:')
print(Z2)

# Step 2: Covariance matrix using Z.T @ Z / (n-1)
C2 = (Z2.T @ Z2) / (X2.shape[0] - 1)
print('\nCovariance Matrix C:')
print(C2)

# Step 3: Eigenvalues
evals2, evecs2 = np.linalg.eig(C2)
idx2   = np.argsort(evals2)[::-1]
evals2 = evals2[idx2]
evecs2 = evecs2[:, idx2]

print('\nEigenvalues:', evals2)
print('Note: λ₂ = 0 means data lies perfectly on a line! 100% variance in 1D.')

# Step 4: Principal component (eigenvector for λ₁=70)
u1 = evecs2[:, 0]
print('\nFirst Eigenvector u₁:', u1)

# Step 5: Project
Y2 = Z2 @ u1
print('\nProjected 1D Values:')
for i in range(3):
    print(f'  Point {i+1}: {Y2[i]:.4f}')

In [ ]:
# ─────────────────────────────────────────
# 📊 VISUALIZATION: Part 2
# ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
colors2 = ['#e74c3c', '#3498db', '#2ecc71']
labels2 = ['(2,11)', '(3,14)', '(7,26)']

for i in range(3):
    ax.scatter(X2[i,0], X2[i,1], color=colors2[i], s=150, zorder=5)
    ax.annotate(labels2[i], (X2[i,0], X2[i,1]),
                textcoords='offset points', xytext=(8,5), fontsize=10)

# Draw the principal component line
t = np.linspace(-4, 4, 100)
pc1_line = mean2[:, None] + np.outer(u1, t)
ax.plot(pc1_line[0], pc1_line[1], 'purple', lw=2, label='PC1 (λ₁=70)')
ax.scatter(*mean2, color='black', s=100, marker='x', zorder=6, label='Mean (4,17)')

ax.set_title('Part 2: Data lies perfectly on PC1 (λ₂=0)', fontsize=12, fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔢 Part 3: Full PCA Pipeline with sklearn (10-Point Dataset)
*(Matches Sections 10–11 of the lecture notes)*

In [ ]:
# ─────────────────────────────────────────
# PART 3: Using sklearn PCA on 10-point dataset
# Covariance matrix from lecture:
#   C = [[0.6166, 0.6154],
#        [0.6154, 0.7166]]
# ─────────────────────────────────────────

# Reconstruct approximate 10-point dataset from lecture sums
# ΣX=18.1, ΣY=19.1, first two rows given
X3 = np.array([
    [2.5, 2.4],
    [0.5, 0.7],
    [2.2, 2.9],
    [1.9, 2.2],
    [3.1, 3.0],
    [2.3, 2.7],
    [2.0, 1.6],
    [1.0, 1.1],
    [1.5, 1.6],
    [1.1, 0.9]
])

# sklearn PCA
pca = SklearnPCA(n_components=1)
X3_reduced = pca.fit_transform(X3)

print('Original shape :', X3.shape)
print('Reduced shape  :', X3_reduced.shape)
print()
print('Explained Variance Ratio:', pca.explained_variance_ratio_)
print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print()
print('Principal Component (eigenvector):', pca.components_)
print()
print('1D Projected Values:')
for i, v in enumerate(X3_reduced):
    print(f'  Point {i+1}: {v[0]:.4f}')

In [ ]:
# ─────────────────────────────────────────
# 📊 VISUALIZATION: Variance Explained Scree Plot
# ─────────────────────────────────────────
pca_full = SklearnPCA(n_components=2)
pca_full.fit(X3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scree Plot
ax1 = axes[0]
ax1.bar(['PC1', 'PC2'], pca_full.explained_variance_ratio_ * 100,
        color=['#3498db', '#e74c3c'], edgecolor='black')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('Scree Plot — Variance per Component', fontweight='bold')
for i, v in enumerate(pca_full.explained_variance_ratio_ * 100):
    ax1.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Original vs Projected
ax2 = axes[1]
ax2.scatter(X3[:,0], X3[:,1], color='#3498db', s=100, label='Original 2D', zorder=5)

# Draw PC1 direction
mean3 = np.mean(X3, axis=0)
pc1v  = pca_full.components_[0]
t = np.linspace(-1.5, 1.5, 100)
line = mean3[:, None] + np.outer(pc1v, t)
ax2.plot(line[0], line[1], 'r-', lw=2, label='PC1 Direction')
ax2.scatter(*mean3, color='black', s=100, marker='x', zorder=6, label='Mean')
ax2.set_title('10-Point Dataset + PC1 Direction', fontweight='bold')
ax2.set_xlabel('X'); ax2.set_ylabel('Y')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## ✏️ Part 4: Practice Problems — Solutions
*(All practice problems from the lecture)*

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 1: Matrix Dimensions')
print('=' * 55)
N_ex, n_feat, p_comp = 500, 20, 3
print(f'Dataset: {N_ex} examples, {n_feat} features, top {p_comp} PCs')
print(f'Covariance Matrix size : {n_feat} × {n_feat}')
print(f'Reduced Dataset size   : {N_ex} × {p_comp}')

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 2: Eigenvalue Significance')
print('=' * 55)
lambdas = np.array([4.5, 0.8, 2.1])
total   = lambdas.sum()
first_pc_idx = np.argmax(lambdas)

print(f'Eigenvalues: {lambdas}')
print(f'First Principal Component = λ{first_pc_idx+1} = {lambdas[first_pc_idx]}')
print(f'Total variance: {total}')

kept  = lambdas[first_pc_idx]
lost  = total - kept
print(f'Variance lost if only PC1 kept: {lost:.1f} / {total:.1f} = {lost/total*100:.1f}%')

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 3: Eigenvector Normalization')
print('=' * 55)
v = np.array([3.0, 4.0])
magnitude = np.linalg.norm(v)
e_norm = v / magnitude

print(f'Raw eigenvector v = {v}')
print(f'Magnitude ||v|| = √(3² + 4²) = √{int(v@v)} = {magnitude}')
print(f'Normalized e  = {e_norm}')
print(f'Verification ||e|| = {np.linalg.norm(e_norm):.4f} (should be 1.0)')

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 4: Covariance Matrix Calc')
print('Dataset: (2,4), (4,8)')
print('=' * 55)
D = np.array([[2,4],[4,8]], dtype=float)
mean_D = D.mean(axis=0)
Dc = D - mean_D
Cov_D = (Dc.T @ Dc) / (D.shape[0] - 1)
print(f'Means: X̄={mean_D[0]}, Ȳ={mean_D[1]}')
print('Covariance Matrix:')
print(Cov_D)

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 5: Eigenvalues of S = [[4,2],[2,3]]')
print('=' * 55)
S_p = np.array([[4,2],[2,3]], dtype=float)
evals_p, evecs_p = np.linalg.eig(S_p)
print(f'Eigenvalues: λ₁={evals_p[0]:.4f}, λ₂={evals_p[1]:.4f}')

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 6: Find normalized eigenvector')
print('M = [[3,1],[1,3]], λ=4')
print('=' * 55)
M = np.array([[3,1],[1,3]], dtype=float)
lam = 4
# (M - λI)v = 0  →  [[−1,1],[1,−1]]v = 0  →  v₁ = v₂
raw_v = np.array([1.0, 1.0])
norm_v = raw_v / np.linalg.norm(raw_v)
print(f'Raw eigenvector: {raw_v}')
print(f'Normalized     : {norm_v}')

In [ ]:
print('=' * 55)
print('PRACTICE PROBLEM 7: Projection')
print('P = [10, 5],  V = [0.8, 0.6]')
print('=' * 55)
P = np.array([10, 5])
V = np.array([0.8, 0.6])
projected = np.dot(P, V)
print(f'1D Value = P · V = {P[0]}×{V[0]} + {P[1]}×{V[1]} = {projected}')

---
## 📝 Summary — The PCA Algorithm

| Step | What We Do | Formula |
|------|-----------|--------|
| 1 | Compute feature means | x̄ᵢ = (1/N)ΣXᵢ |
| 2 | Compute covariance matrix S | Cov(i,j) = 1/(N-1) Σ(xᵢ-x̄ᵢ)(xⱼ-x̄ⱼ) |
| 3 | Find eigenvalues λ | det(S - λI) = 0 |
| 4 | Find & normalize eigenvectors | e = V / ‖V‖ |
| 5 | Sort by eigenvalue (↓) | λ₁ ≥ λ₂ ≥ … ≥ λₙ |
| 6 | Select top p components | W = [e₁, e₂, …, eₚ] |
| 7 | Project original data | Y = (X - X̄) × W |

---
*End of Lecture 22 — PCA Notebook*